# ETS Benchmark, the flavours of the paper: smooth 1.3.0 (2026-11-05)

Rerun of the ETS rows of "Smooth forecasting with the smooth package in R/Python"
(M1, M3 and Tourism, 5,315 series) on the Python implementation of **smooth 1.3.0**:
`ADAM ETS` with backcasting and `ES` with backcasting and optimal initialisation (`model="ZXZ"`, `ZXN` on non-seasonal data).

**Only Python smooth is fitted.** No R and no other packages: the Nemenyi section reads their
columns from `smooth-paper/Data`, as they were for the paper (smooth 1.0.8).

The harness follows `smooth-paper/run_full_benchmark.py`: one fit per (method, series) gives
the point forecast and the 99 quantiles (`interval="prediction"`, `side="both"`, the point
forecast at 0.5), the numerical libraries are pinned to one thread, the budget is 120 s per
series, and the methods run one after another in pools of 30 workers on the AMD Ryzen 9 3950X.

In [1]:
# Pin the numerical libraries before they are imported: a pinned time is the cost of one
# series on one core, and the worker pools do not oversubscribe the machine.
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"

import time, signal, subprocess, sys, warnings, multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
from importlib.metadata import version
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

HERE = os.getcwd()
DATA = os.path.join(HERE, "smooth-paper", "Data")
LEVELS = np.round(np.arange(0.01, 1.0, 0.01), 2)               # 99 quantile levels
TWO = [round(1 - 2 * t, 2) for t in LEVELS if t < 0.5]          # two-sided interval levels
DATE, VERSION_TAG, KIND = "2026-11-05", "smooth-1-3-0", "ets"
TIMEOUT = 120                                                   # per series, as in the paper
WORKERS = int(os.environ.get("BENCH_WORKERS", "30"))
LIMIT = int(os.environ.get("BENCH_LIMIT", "0"))                 # >0: a smoke test, nothing is saved
RESULTS = os.path.join(HERE, f"{DATE}-benchmark-{KIND}-{VERSION_TAG}.npy")
COVERAGE = os.path.join(HERE, f"{DATE}-benchmark-{KIND}-{VERSION_TAG}-coverage.npy")
MODELS = os.path.join(HERE, f"{DATE}-benchmark-{KIND}-{VERSION_TAG}-models.csv")

## Versions

In [2]:
print(f"python {sys.version.split()[0]}   cores {multiprocessing.cpu_count()}   workers {WORKERS}")
print(sys.executable)
for _p in ("smooth", "greybox", "numpy", "pandas", "fcompdata"):
    print(f"{_p:12s}v{version(_p)}")
print("git HEAD:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())
print("modified tracked files:", subprocess.run(["git", "status", "--porcelain", "--untracked-files=no"],
                                                capture_output=True, text=True).stdout.count("\n"))

python 3.14.4   cores 32   workers 30
/home/config/Misc/Python/Libraries/openforecast-org/smooth/python/.venv/bin/python3
smooth      v1.3.0
greybox     v1.0.11
numpy       v2.3.5
pandas      v2.3.3
fcompdata   v0.1.4
git HEAD: b18cda63 Merge pull request #439 from openforecast-org/code-optimisation
modified tracked files: 0


## Datasets

In [3]:
from fcompdata import M1, M3, Tourism

labelled = ([("M1", k, M1[k]) for k in M1.keys()] + [("M3", k, M3[k]) for k in M3.keys()]
            + [("Tourism", k, Tourism[k]) for k in Tourism.keys()])
datasets = [s for _, _, s in labelled]
ids = [f"{d}-{k}" for d, k, _ in labelled]
print(f"M1: {len(M1)}   M3: {len(M3)}   Tourism: {len(Tourism)}   total: {len(datasets)}")
if LIMIT:
    datasets, ids = datasets[:LIMIT], ids[:LIMIT]
    print(f"SMOKE TEST: the first {LIMIT} series, nothing is saved")

M1: 1001   M3: 3003   Tourism: 1311   total: 5315


## Methods and harness

In [4]:
METHODS = [
    ("ADAM ETS Back", {"class": "ADAM", "initial": "backcasting"}),
    ("ES Back", {"class": "ES", "initial": "backcasting"}),
    ("ES Opt", {"class": "ES", "initial": "optimal"}),
]


def fit_predict(cfg, s):
    from smooth import ADAM, ES
    model = "ZXZ" if s.period > 1 else "ZXN"
    m = (ADAM if cfg["class"] == "ADAM" else ES)(model=model, lags=lags_of(s), initial=cfg["initial"])
    m.fit(np.asarray(s.x, float))
    mean, Q = quantiles(m.predict(h=s.h, interval="prediction", level=TWO, side="both"))
    return m.model_name, mean, Q

In [5]:
def lags_of(s):
    return [1, s.period] if s.period > 1 else [1]


def quantiles(fc):
    # The 99 quantiles from the two-sided intervals, with the point forecast at 0.5.
    mean = np.asarray(fc.mean, float).ravel()
    lower, upper = np.asarray(fc.lower, float), np.asarray(fc.upper, float)
    lo = {round(float(c), 2): i for i, c in enumerate(fc.lower.columns)}
    up = {round(float(c), 2): i for i, c in enumerate(fc.upper.columns)}
    Q = np.empty((mean.size, 99))
    for k, t in enumerate(LEVELS):
        if abs(t - 0.5) < 1e-9:
            Q[:, k] = mean
        elif t < 0.5:
            Q[:, k] = lower[:, lo[round(t, 2)]]
        else:
            Q[:, k] = upper[:, up[round(t, 2)]]
    return mean, Q


def score(s, mean, Q):
    # As smooth-paper/run_full_benchmark.py: RMSSE, the scaled pinball over the 99 sorted
    # quantiles and the coverage of each, plus SAME.
    x, y = np.asarray(s.x, float), np.asarray(s.xx, float)
    scale_sq, scale_abs = np.mean(np.diff(x) ** 2), np.mean(np.abs(np.diff(x)))
    rmsse = np.sqrt(np.mean((y - mean) ** 2) / scale_sq) if scale_sq else np.nan
    same = np.abs(np.mean(y - mean)) / scale_abs if scale_abs else np.nan
    if not np.all(np.isfinite(Q)):
        return rmsse, same, np.nan, np.full(99, np.nan)
    Q = np.sort(Q, axis=1)
    diff = y[:, None] - Q
    pinball = np.maximum(LEVELS * diff, (LEVELS - 1) * diff).mean(axis=0) / (scale_abs if scale_abs else 1.0)
    return rmsse, same, pinball.mean(), (y[:, None] <= Q).mean(axis=0)


class Timeout(Exception):
    pass


def _alarm(signum, frame):
    raise Timeout()


def task(args):
    # One (method, series) fit: rmsse, same, time, pinball, timed out; the coverage; the model.
    warnings.filterwarnings("ignore")
    j, i, s, cfg = args
    out, cov, name = np.full(5, np.nan), np.full(99, np.nan), ""
    out[4] = 0.0
    signal.signal(signal.SIGALRM, _alarm)
    signal.alarm(TIMEOUT)
    try:
        start = time.time()
        model, mean, Q = fit_predict(cfg, s)
        out[2] = time.time() - start
        signal.alarm(0)
        name = model
        out[0], out[1], out[3], cov = score(s, mean[: s.h], Q[: s.h])
    except Timeout:
        out[4] = 1.0
    except Exception as e:
        name = f"ERROR {type(e).__name__}: {e}"[:200]
    finally:
        signal.alarm(0)
    return j, i, out, cov, name


def run(methods, workers):
    # The methods run one after another, each in its own pool, so the times do not mix them.
    result = np.full((len(methods), len(datasets), 5), np.nan)
    coverage = np.full((len(methods), len(datasets), 99), np.nan)
    names = [[""] * len(datasets) for _ in methods]
    ctx = multiprocessing.get_context("fork")
    for j, (label, cfg) in enumerate(methods):
        start = time.time()
        with ProcessPoolExecutor(max_workers=workers, mp_context=ctx) as ex:
            futures = [ex.submit(task, (j, i, s, cfg)) for i, s in enumerate(datasets)]
            for f in as_completed(futures):
                _, i, out, cov, name = f.result()
                result[j, i], coverage[j, i], names[j][i] = out, cov, name
        print(f"{label:20s} {(time.time() - start) / 60:6.1f} min  "
              f"failed {int(np.isnan(result[j, :, 0]).sum())}  timeouts {int(result[j, :, 4].sum())}", flush=True)
    return result, coverage, names

## Run or load

In [6]:
# Fit, or load the saved run (BENCH_RERUN=1 forces a refit).
# current: (methods, series, [rmsse, same, time, pinball, timed out]); coverage: (methods, series, 99).
if LIMIT or os.environ.get("BENCH_RERUN", "0") == "1" or not os.path.exists(RESULTS):
    print(f"{len(METHODS)} methods x {len(datasets)} series, {WORKERS} workers, {TIMEOUT}s per series")
    current, coverage, names = run(METHODS, WORKERS)
    models = pd.DataFrame({m: n for (m, _), n in zip(METHODS, names)}, index=ids)
    if not LIMIT:
        np.save(RESULTS, current)
        np.save(COVERAGE, coverage)
        models.to_csv(MODELS)
else:
    current, coverage = np.load(RESULTS), np.load(COVERAGE)
    models = pd.read_csv(MODELS, index_col=0)
    print("loaded", os.path.basename(RESULTS), current.shape)
errors = models.stack()[lambda v: v.astype(str).str.startswith("ERROR")]
print(f"errors: {len(errors)}")
print(errors.value_counts().head(5).to_string() if len(errors) else "")

3 methods x 5315 series, 30 workers, 120s per series


ADAM ETS Back           0.5 min  failed 0  timeouts 0


ES Back                 0.5 min  failed 0  timeouts 0


ES Opt                  1.7 min  failed 0  timeouts 0


errors: 0



## Results, in the columns of the paper

In [7]:
# The columns of the paper's table. MCE is computed as for that table: the coverage is
# averaged over the series at each level (it reproduces the paper's values from its saved arrays).
def row(rmsse, pinball, cov, seconds):
    ok = np.isfinite(cov).all(axis=1)
    return {"Median RMSSE": np.nanmedian(rmsse), "Mean RMSSE": np.nanmean(rmsse),
            "Median pinball": np.nanmedian(pinball), "Mean pinball": np.nanmean(pinball),
            "MCE": np.abs(cov[ok].mean(axis=0) - LEVELS).mean(), "Time, s": np.nanmean(seconds)}


labels = [m for m, _ in METHODS]
table = pd.DataFrame([row(current[j, :, 0], current[j, :, 3], coverage[j], current[j, :, 2])
                      for j in range(len(labels))], index=labels)
table["Failed"] = np.isnan(current[:, :, 0]).sum(axis=1)
table["Timeouts"] = current[:, :, 4].sum(axis=1).astype(int)
table["Mean SAME"] = np.nanmean(current[:, :, 1], axis=1)
print(f"smooth {version('smooth')}, {len(datasets)} series (lower is better)")
print(table.to_string(float_format=lambda v: f"{v:.3f}"))

smooth 1.3.0, 5315 series (lower is better)
               Median RMSSE  Mean RMSSE  Median pinball  Mean pinball   MCE  Time, s  Failed  Timeouts  Mean SAME
ADAM ETS Back         1.187       1.946           0.505         0.859 0.053    0.180       0         0      1.967
ES Back               1.188       1.947           0.511         0.849 0.044    0.158       0         0      1.967
ES Opt                1.190       1.948           0.510         0.848 0.041    0.586       0         0      1.968


## Against the paper (smooth 1.0.8)

In [8]:
PAPER_POINT, PAPER_DIST = "2026-08-24-benchmark-ets-point.npy", "2026-08-24-benchmark-ets-dist.npy"
PAPER_INDEX = [0, 1, 2]     # the positions of these methods in the saved arrays
CSV_NAMES = ['ADAM ETS back', 'ES back', 'ES opt']
# The Python arrays behind the paper (smooth 1.0.8), same harness, same machine type.
n = len(datasets)
paper_point = np.load(PAPER_POINT)[:, :n]
paper_dist = np.load(PAPER_DIST)[:, :n]
rows, changes = {}, []
for j, (label, k) in enumerate(zip(labels, PAPER_INDEX)):
    old_r, old_p = paper_point[k, :, 0], np.nanmean(paper_dist[k, :, :, 0], axis=1)
    rows[(label, "1.0.8")] = row(old_r, old_p, paper_dist[k, :, :, 1], paper_point[k, :, 1])
    rows[(label, version("smooth"))] = table.loc[label, list(rows[(label, "1.0.8")])].to_dict()
    for metric, old, new in (("RMSSE", old_r, current[j, :, 0]), ("Pinball", old_p, current[j, :, 3])):
        d = new - old
        changes.append({"Method": label, "Metric": metric, "Changed": np.mean(np.abs(d) > 1e-6),
                        "Better": np.mean(d < -1e-6), "Worse": np.mean(d > 1e-6)})
print(pd.DataFrame(rows).T.to_string(float_format=lambda v: f"{v:.3f}"))
print()
print(pd.DataFrame(changes).to_string(index=False, float_format=lambda v: f"{v:.1%}"))

                     Median RMSSE  Mean RMSSE  Median pinball  Mean pinball   MCE  Time, s
ADAM ETS Back 1.0.8         1.188       1.953           0.505         0.863 0.053    0.168
              1.3.0         1.187       1.946           0.505         0.859 0.053    0.180
ES Back       1.0.8         1.190       1.947           0.513         0.849 0.044    0.159
              1.3.0         1.188       1.947           0.511         0.849 0.044    0.158
ES Opt        1.0.8         1.190       1.948           0.510         0.848 0.041    0.591
              1.3.0         1.190       1.948           0.510         0.848 0.041    0.586

       Method  Metric  Changed  Better  Worse
ADAM ETS Back   RMSSE    26.0%   12.6%  13.4%
ADAM ETS Back Pinball    45.2%   21.8%  23.4%
      ES Back   RMSSE    26.9%   13.0%  13.9%
      ES Back Pinball    45.6%   26.4%  19.2%
       ES Opt   RMSSE     0.0%    0.0%   0.0%
       ES Opt Pinball    23.5%   16.1%   7.4%


## Selected models

In [9]:
# The selected models, the share of the series with each family of structures.
print(pd.DataFrame({m: models[m].value_counts(normalize=True).head(8) for m in models}).fillna(0)
      .sort_values(models.columns[0], ascending=False).head(12).to_string(float_format=lambda v: f"{v:.1%}"))
print()
for j, label in enumerate(labels):
    if j:
        print(f"same model as {labels[0]} for {label}: {(models[label] == models[labels[0]]).mean():.1%}")

           ADAM ETS Back  ES Back  ES Opt
ETS(MNN)           19.0%    17.3%   19.4%
ETS(MAN)           16.4%    15.8%   16.6%
ETS(MNM)           15.7%    15.6%    8.5%
ETS(ANN)           13.2%    14.8%   13.3%
ETS(AAN)            9.1%     9.2%    7.9%
ETS(MAM)            8.2%     8.4%   11.9%
ETS(ANA)            8.1%     7.6%    5.7%
ETS(AAA)            2.8%     0.0%    0.0%
ETS(MAdN)           0.0%     2.7%    3.3%

same model as ADAM ETS Back for ES Back: 86.4%
same model as ADAM ETS Back for ES Opt: 64.7%


## Nemenyi mean ranks against the saved columns

The competitors are not refitted.

In [10]:
# Mean ranks with the rows of smooth replaced, as in the paper's Nemenyi test. The other columns are
# read from the saved files of the paper and are NOT refitted, so they stay at their versions.
Q95 = {5: 2.7278, 7: 2.9483, 8: 3.0309}     # qtukey(0.95, k, Inf) / sqrt(2)


def nemenyi(metric, column):
    other = pd.concat([pd.read_csv(os.path.join(DATA, f"{side}-{KIND}-{metric}.csv")) for side in ("r", "python")],
                      axis=1).dropna(axis=1, how="all").iloc[:len(datasets)]
    errors = other.copy()
    for label, csv_name in zip(labels, CSV_NAMES):
        errors[csv_name] = current[labels.index(label), :, column]
    errors = errors.dropna()
    k, N = errors.shape[1], len(errors)
    cd = Q95[k] * np.sqrt(k * (k + 1) / (6 * N))
    ranks = errors.rank(axis=1).mean().sort_values()
    print(f"{metric}: {k} methods, {N} series, critical distance {cd:.3f}")
    print(pd.DataFrame({"Mean rank": ranks, "In the top group": ranks - ranks.iloc[0] <= cd}).to_string(
        float_format=lambda v: f"{v:.3f}"))


nemenyi("rmsse", 0)
print()
nemenyi("pinball", 3)

rmsse: 8 methods, 5315 series, critical distance 0.144
                       Mean rank  In the top group
statsforecast AutoETS      4.345              True
ADAM ETS back              4.345              True
forecast ets               4.356              True
ES back                    4.371              True
sktime AutoETS             4.373              True
ES opt                     4.380              True
aeon AutoETS               4.707             False
skforecast AutoETS         5.122             False

pinball: 7 methods, 5315 series, critical distance 0.124
                       Mean rank  In the top group
ES opt                     3.835              True
ES back                    3.887              True
forecast ets               3.912              True
sktime AutoETS             3.921              True
statsforecast AutoETS      3.938              True
ADAM ETS back              3.965             False
skforecast AutoETS         4.542             False


## Findings

* **Accuracy barely moves from smooth 1.0.8.** Mean RMSSE of ADAM ETS backcasting goes from 1.953 to 1.946, median from 1.188 to 1.187. ES backcasting: median 1.190 to 1.188. ES optimal: the RMSSE is unchanged on all the series. MCE is unchanged to three decimals.
* **The RMSSE conclusion holds:** the six implementations stay in one Nemenyi group, with aeon and skforecast outside it.
* **On the pinball, ADAM ETS backcasting leaves the top group by a small margin:** its mean rank of 3.965 is 0.130 behind ES optimal, against a critical distance of 0.124. ES optimal now has the best rank. The competitor columns are the saved ones of the paper.
* No failures and no timeouts.